<a href="https://colab.research.google.com/github/bala-baskar/LLM_foundations/blob/main/tech_books/Build_a_Large_Language_Model_(From_Scratch)/practice_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
os.chdir("/content/LLM_foundations/tech_books/Build_a_Large_Language_Model_(From_Scratch)")

In [3]:
!ls

build_large_language_model_part1.ipynb	build_llm_part1.ipynb  practice_1.ipynb
build_large_language_model_part2.ipynb	build_llm_part2.ipynb  src


### 1. Download data

In [6]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = os.path.join("data","the-verdict.txt")
urllib.request.urlretrieve(url, file_path)

('data/the-verdict.txt', <http.client.HTTPMessage at 0x7d1d86772540>)

In [7]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### 2. Create Pytorch Dataset

In [8]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        encoded_token_ids = tokenizer.encode(text)

        for id in range(0,len(encoded_token_ids)-max_length,stride):
            input_chunk = encoded_token_ids[id:id+max_length]
            target_chunk = encoded_token_ids[id+1:id+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.input_ids)

In [9]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = GPTDataset(text=raw_text,
                           tokenizer=tokenizer,
                           max_length=5,
                           stride=5)

In [10]:
train_dataloader = DataLoader(train_dataset,
                              batch_size=4,
                              shuffle=False,
                              drop_last=False,
                              num_workers=0)

In [11]:
first_batch = next(iter(train_dataloader))
print(first_batch)

[tensor([[   40,   367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899,  2138],
        [  257,  7026, 15632,   438,  2016],
        [  257,   922,  5891,  1576,   438]]), tensor([[  367,  2885,  1464,  1807,  3619],
        [  402,   271, 10899,  2138,   257],
        [ 7026, 15632,   438,  2016,   257],
        [  922,  5891,  1576,   438,   568]])]


In [21]:
inputs_1, target_1 = first_batch
print(inputs_1.shape, target_1.shape)

torch.Size([4, 5]) torch.Size([4, 5])


### 3. Embedding layer

In [14]:
vocab_size = tokenizer.n_vocab
embed_size = 256
print("Vocab size of tokenizer:",vocab_size)
print("Embedding size of input tokens:",embed_size)

Vocab size of tokenizer: 50257
Embedding size of input tokens: 256


In [24]:
embedding_layer = torch.nn.Embedding(vocab_size,embed_size)
embedding_layer(inputs_1).shape

torch.Size([4, 5, 256])

In [25]:
# Add positional embedding
context_length = inputs_1.shape[1] # max_length
print("Context length:",context_length)
pos_embedding_layer = torch.nn.Embedding(context_length,embed_size)
pos_embedding_layer(torch.arange(context_length)) # assign ordinality to positions

Context length: 5


tensor([[ 1.1331,  1.2029,  0.6631,  ...,  0.1322,  0.2075,  0.4155],
        [ 0.1972, -0.2927,  0.0924,  ...,  1.3603,  0.5619, -1.0479],
        [ 0.3780, -1.6069,  0.4085,  ...,  1.3259, -1.4764,  0.3270],
        [ 0.1733,  0.1210,  1.0877,  ..., -2.7757, -0.3588, -0.7249],
        [ 0.0753, -2.0378, -0.1952,  ..., -0.5517,  1.2256, -0.0517]],
       grad_fn=<EmbeddingBackward0>)

In [29]:
# embedding layer
input_embedding = embedding_layer(inputs_1) + pos_embedding_layer(torch.arange(inputs_1.shape[1]))
input_embedding.shape

torch.Size([4, 5, 256])

### 4. Self Attention

In [30]:
input_embedding @ input_embedding.T

/tmp/ipykernel_2111/4150772768.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  input_embedding @ input_embedding.T


RuntimeError: The size of tensor a (4) must match the size of tensor b (256) at non-singleton dimension 0

In [32]:
input_embedding[0,:,:] @ input_embedding[0,:,:].T

tensor([[ 4.9529e+02,  1.2618e+01,  2.6070e-01,  3.8938e+01,  2.6246e+00],
        [ 1.2618e+01,  4.9766e+02, -3.1491e+00,  4.9578e+01, -1.7987e+01],
        [ 2.6070e-01, -3.1491e+00,  4.8177e+02, -1.9435e+01,  2.1048e+01],
        [ 3.8938e+01,  4.9578e+01, -1.9435e+01,  5.1923e+02,  5.1337e+01],
        [ 2.6246e+00, -1.7987e+01,  2.1048e+01,  5.1337e+01,  5.1968e+02]],
       grad_fn=<MmBackward0>)

In [40]:
tokenizer.decode(inputs_1[0].tolist())

'I HAD always thought'